In [ ]:
import cv2
import numpy as np
import os


folders = ["gray", "median", "log", "contrast", "enhanced"]
for f in folders:
    os.makedirs(f, exist_ok=True)


def enhance_and_save(frame, frame_id):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(f"gray/frame_{frame_id}.jpg", gray)

    median = cv2.medianBlur(gray, 3)
    cv2.imwrite(f"median/frame_{frame_id}.jpg", median)

    median_float = median.astype(np.float32)

    r_max = max(np.max(median_float), 1)
    c = 255 / np.log1p(r_max)
    log_img = c * np.log1p(median_float)

    log_img = np.nan_to_num(log_img, nan=0, posinf=255, neginf=0)
    log_img = np.clip(log_img, 0, 255).astype(np.uint8)
    cv2.imwrite(f"log/frame_{frame_id}.jpg", log_img)

    r_min = np.min(log_img)
    r_max = np.max(log_img)

    if r_max - r_min == 0:
        contrast = log_img
    else:
        contrast = (log_img - r_min) * (255 / (r_max - r_min))

    contrast = np.clip(contrast, 0, 255).astype(np.uint8)
    cv2.imwrite(f"contrast/frame_{frame_id}.jpg", contrast)

    enhanced = cv2.equalizeHist(contrast)
    cv2.imwrite(f"enhanced/frame_{frame_id}.jpg", enhanced)

    return enhanced



video = cv2.VideoCapture("final_video.mp4")

frame_count = 0
MAX_FRAMES = 30 

while True:
    ret, frame = video.read()
    if not ret:
        break

    enhanced = enhance_and_save(frame, frame_count)

    cv2.imshow("Enhanced Output", enhanced)

    if cv2.waitKey(30) & 0xFF in [27, ord('q')]:
        break

    frame_count += 1

video.release()
cv2.destroyAllWindows()

print("All stages saved successfully!")

All stages saved successfully!


In [3]:
import matplotlib.pyplot as plt


images={}

stages = ["gray", "median", "log", "contrast", "enhanced"]


for stage in stages:
    path = f"{stage}/frame_22.jpg"
    img = cv2.imread(path, 0)

    if img is None:
        print(f"Error loading {path}")
        continue

    images[stage] = img


plt.figure(figsize=(12, 6))

for i, (name, img) in enumerate(images.items()):
    plt.subplot(2, 5, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(name.upper())
    plt.axis('off')

for i, (name, img) in enumerate(images.items()):
    plt.subplot(2, 5, i+6)
    plt.hist(img.ravel(), bins=256)
    plt.title(f"{name} Hist")

plt.tight_layout()
plt.show()

Error loading gray/frame_22.jpg
Error loading median/frame_22.jpg
Error loading log/frame_22.jpg
Error loading contrast/frame_22.jpg
Error loading enhanced/frame_22.jpg


<Figure size 1200x600 with 0 Axes>